In [0]:
%sql
-- Upload claims.csv to a Unity Catalog Volume first, then update the path below
-- Example: /Volumes/main/default/volume_name/claims.csv


SELECT * 
FROM read_files(
  '/Volumes/main/default/raw_data/claims.csv',
  format => 'csv',
  header => true
)

claim_id,policy_id,customer_id,claim_date,close_date,claim_amount,claim_status,city,claim_type,_rescued_data
C001,P001,U001,2026-01-05,2026-01-10,2500,CLOSED,Quito,Auto,null
C002,P002,U002,2026-01-08,2026-01-20,8000,CLOSED,Guayaquil,Salud,null
C003,P003,U003,2026-01-10,2026-01-25,1500,CLOSED,Cuenca,Hogar,null
C004,P001,U001,2026-02-01,2026-02-12,12000,CLOSED,Quito,Auto,null
C005,P004,U004,2026-02-03,2026-02-10,500,CLOSED,Manta,Viaje,null
C006,P005,U005,2026-02-10,2026-02-18,30000,CLOSED,Quito,Empresarial,null
C007,P002,U006,2026-02-15,2026-02-28,2000,CLOSED,Loja,Salud,null
C008,P006,U007,2026-03-01,2026-03-08,950,CLOSED,Ambato,Hogar,null
C009,P007,U008,2026-03-03,2026-03-20,22000,CLOSED,Quito,Auto,null
C010,P003,U009,2026-03-05,2026-03-14,4000,CLOSED,Cuenca,Hogar,null


In [0]:
%sql

CREATE VOLUME IF NOT EXISTS main.default.raw_data;

# 🚀 PASO 8 — Cargar Claims

In [0]:
claims = spark.read.csv(
    "/Volumes/main/default/raw_data/claims.csv",
    header=True,
    inferSchema=True
)

display(claims)

claim_id,policy_id,customer_id,claim_date,close_date,claim_amount,claim_status,city,claim_type
C001,P001,U001,2026-01-05,2026-01-10,2500,CLOSED,Quito,Auto
C002,P002,U002,2026-01-08,2026-01-20,8000,CLOSED,Guayaquil,Salud
C003,P003,U003,2026-01-10,2026-01-25,1500,CLOSED,Cuenca,Hogar
C004,P001,U001,2026-02-01,2026-02-12,12000,CLOSED,Quito,Auto
C005,P004,U004,2026-02-03,2026-02-10,500,CLOSED,Manta,Viaje
C006,P005,U005,2026-02-10,2026-02-18,30000,CLOSED,Quito,Empresarial
C007,P002,U006,2026-02-15,2026-02-28,2000,CLOSED,Loja,Salud
C008,P006,U007,2026-03-01,2026-03-08,950,CLOSED,Ambato,Hogar
C009,P007,U008,2026-03-03,2026-03-20,22000,CLOSED,Quito,Auto
C010,P003,U009,2026-03-05,2026-03-14,4000,CLOSED,Cuenca,Hogar


# 🚀 PASO 9 — Guardar Delta Table


In [0]:
from pyspark.sql.functions import col

claims.withColumn("claim_amount", col("claim_amount").cast("double")) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sdp_catalog.bronze.claims_raw")

# 🚀 PASO 10 — Customers

In [0]:
customers = spark.read.csv(
    "/Volumes/main/default/raw_data/customers.csv",
    header=True,
    inferSchema=True
)

customers.write.format("delta") \
.mode("overwrite") \
.saveAsTable("sdp_catalog.bronze.customers_raw")

# 🚀 PASO 11 — Policies

In [0]:
policies = spark.read.csv(
    "/Volumes/main/default/raw_data/policies.csv",
    header=True,
    inferSchema=True
)

policies.withColumn("premium_amount", col("premium_amount").cast("double")) \
    .write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("sdp_catalog.bronze.policies_raw")

# 🚀 PASO 12 - Ver tablas Bronze

In [0]:
display(
    spark.sql("SELECT * FROM sdp_catalog.bronze.claims_raw")
)

claim_id,policy_id,customer_id,claim_date,close_date,claim_amount,claim_status,city,claim_type
C001,P001,U001,2026-01-05,2026-01-10,2500.0,CLOSED,Quito,Auto
C002,P002,U002,2026-01-08,2026-01-20,8000.0,CLOSED,Guayaquil,Salud
C003,P003,U003,2026-01-10,2026-01-25,1500.0,CLOSED,Cuenca,Hogar
C004,P001,U001,2026-02-01,2026-02-12,12000.0,CLOSED,Quito,Auto
C005,P004,U004,2026-02-03,2026-02-10,500.0,CLOSED,Manta,Viaje
C006,P005,U005,2026-02-10,2026-02-18,30000.0,CLOSED,Quito,Empresarial
C007,P002,U006,2026-02-15,2026-02-28,2000.0,CLOSED,Loja,Salud
C008,P006,U007,2026-03-01,2026-03-08,950.0,CLOSED,Ambato,Hogar
C009,P007,U008,2026-03-03,2026-03-20,22000.0,CLOSED,Quito,Auto
C010,P003,U009,2026-03-05,2026-03-14,4000.0,CLOSED,Cuenca,Hogar


In [0]:
display(
    spark.sql("SELECT * FROM sdp_catalog.bronze.policies_raw")
)

policy_id,policy_type,premium_amount,start_date,end_date
P001,Auto,1200.0,2025-01-01,2026-01-01
P002,Salud,2500.0,2025-01-01,2026-01-01
P003,Hogar,900.0,2025-01-01,2026-01-01
P004,Viaje,300.0,2025-01-01,2026-01-01
P005,Empresarial,10000.0,2025-01-01,2026-01-01
P006,Hogar,850.0,2025-01-01,2026-01-01
P007,Auto,1500.0,2025-01-01,2026-01-01
